# 05 — Clean NWSL Media Coverage Data

Loads the raw MediaCloud pull and filters it down to articles actually relevant to the NWSL: keyword-matches on title (league name, team names/nicknames, "women's soccer"), then deduplicates and drops unused columns.

**Inputs:** `data/raw/nwsl_articles_raw.csv`
**Outputs:** `data/processed/nwsl_articles_filtered.csv`


In [ ]:
import pandas as pd
import os

DATA_RAW_DIR = os.path.join("..", "data", "raw")
DATA_PROCESSED_DIR = os.path.join("..", "data", "processed")
os.makedirs(DATA_PROCESSED_DIR, exist_ok=True)

KEYWORDS = [
    "NWSL",
    "National Women's Soccer League",
    "women's soccer",
    "Angel City",
    "OL Reign",
    "Portland Thorns",
    "North Carolina Courage",
    "Washington Spirit",
    "Chicago Red Stars",
    "Houston Dash",
    "Gotham FC",
    "Orlando Pride",
    "Racing Louisville FC",
    "San Diego Wave",
    "Boston Legacy FC",
    "Denver Summit FC",
    "Sky Blue FC",
    "Boston Breakers",
]


def load_and_filter_articles(path):
    """Load the raw article CSV and filter down to NWSL-relevant rows."""
    media = pd.read_csv(path)
    print(f"Total articles before filtering: {len(media)}")

    media["publish_date"] = pd.to_datetime(media["publish_date"])

    pattern = "|".join(KEYWORDS)
    media = media[media["title"].str.contains(pattern, case=False, na=False)]
    media = media.drop_duplicates(subset=["title", "publish_date"])
    media = media.drop(columns=["description"], errors="ignore")

    print(f"Total articles after filtering: {len(media)}")
    return media


In [ ]:
media = load_and_filter_articles(os.path.join(DATA_RAW_DIR, "nwsl_articles_raw.csv"))

out_path = os.path.join(DATA_PROCESSED_DIR, "nwsl_articles_filtered.csv")
media[["publish_date", "media_name", "title", "url"]].to_csv(out_path, index=False)
print(f"Saved {len(media)} rows to nwsl_articles_filtered.csv")
